# [2.2] Vanilla Policy Gradient - my solutions

![policy gradient](https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png)

Working through the VPG exercises. The core idea: maximise $J(	heta)=\mathbb{E}_	au[G(	au)]$. Using $
abla p = p
abla\log p$, the environment terms drop out (no $	heta$), so

$$
abla_	heta J=\mathbb{E}_	au\Big[\sum_t 
abla_	heta\log\pi_	heta(a_t|s_t)\,G(	au)\Big]$$

Below: the helper functions, then REINFORCE on CartPole.

### `compute_returns`
Discounted return per (env, step), reset whenever an episode ends. Working backwards: $G_i = r_i + \gamma G_{i+1}(1-	ext{done}_i)$.

In [1]:
import torch as t, torch.nn.functional as F

def compute_returns(rewards, done, gamma=0.9):
    ne, ns = rewards.shape
    returns = t.zeros_like(rewards)
    G = t.zeros_like(rewards[:, 0])
    for i in reversed(range(ns)):
        G = rewards[:, i] + gamma * G * (~done[:, i])   # mask resets G at episode boundary
        returns[:, i] = G
    return returns

# check against the example in the exercise: rewards/done = [0,0,1,0,1] -> [g^2,g,1,g,1]
g = 0.9
print(compute_returns(t.tensor([[0.,0,1,0,1]]), t.tensor([[0,0,1,0,1]], dtype=t.bool), g))

tensor([[0.8100, 0.9000, 1.0000, 0.9000, 1.0000]])


### `compute_logprobs_and_entropy`
Pick out $\log\pi(a_t|s_t)$ for the taken actions, and the per-step entropy of the policy.

In [2]:
def compute_logprobs_and_entropy(logits, actions):
    logp = F.log_softmax(logits, dim=-1)
    ne, ns = actions.shape
    ei = t.arange(ne)[:, None]; si = t.arange(ns)[None, :]
    logp_taken = logp[ei, si, actions]
    entropy = -(logp.exp() * logp).sum(dim=-1)
    return logp_taken, entropy

# uniform logits over 4 actions -> entropy should be log(4)
_, ent = compute_logprobs_and_entropy(t.zeros(1,1,4), t.zeros(1,1, dtype=t.long))
print('entropy', ent.item(), 'vs log4', t.log(t.tensor(4.)).item())

entropy 1.3862943649291992 vs log4 1.3862943649291992


### `normalize_returns`, `compute_importance_weights`, `compute_reinforce_loss`
Normalise returns (cheap baseline). Importance weights $\exp(\log\pi_	ext{new}-\log\pi_	ext{old})$, detached, optionally clipped. Loss = mean of $iw\cdot\log\pi\cdot$advantage (negate for a descent optimiser).

In [3]:
def normalize_returns(r):
    return (r - r.mean()) / (r.std() + 1e-8)

def compute_importance_weights(new_lp, old_lp, clip=None):
    iw = t.exp(new_lp - old_lp).detach()
    return t.clamp(iw, 1 - clip, 1 + clip) if clip is not None else iw

def compute_reinforce_loss(returns, logp_taken, iw):
    adv = returns - returns.mean(0, keepdim=True)   # per-timestep baseline across envs
    return -(iw * logp_taken * adv.detach()).mean()

# clip sanity: iw=exp(5) clipped to 1+0.2
print('clipped iw', compute_importance_weights(t.tensor([[5.]]), t.tensor([[0.]]), 0.2).item())

clipped iw 1.2000000476837158


### Putting it together: REINFORCE on CartPole
Reward-to-go weighting + normalised advantage. Should climb to ~500.

In [4]:
!pip -q install gymnasium
import gymnasium as gym, torch.nn as nn
from torch.distributions import Categorical
t.manual_seed(0)
pi = nn.Sequential(nn.Linear(4,64), nn.Tanh(), nn.Linear(64,64), nn.Tanh(), nn.Linear(64,2))
opt = t.optim.Adam(pi.parameters(), 1e-2)
env = gym.make('CartPole-v1')

def rtg(rs):
    out=[0.]*len(rs); run=0.
    for i in reversed(range(len(rs))): run=rs[i]+run; out[i]=run
    return out

for ep in range(60):
    obs,act,w,rets=[],[],[],[]; steps=0
    while steps<2000:
        o,_=env.reset(); er=[]; d=False
        while not d:
            a=Categorical(logits=pi(t.as_tensor(o,dtype=t.float32))).sample().item()
            obs.append(o); act.append(a)
            o,r,te,tr,_=env.step(a); d=te or tr; er.append(r); steps+=1
        w+=rtg(er); rets.append(sum(er))
    logp=Categorical(logits=pi(t.as_tensor(obs,dtype=t.float32))).log_prob(t.as_tensor(act))
    wt=normalize_returns(t.as_tensor(w,dtype=t.float32))
    loss=-(logp*wt).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if ep%10==0 or ep==59: print(f'epoch {ep:2d}  avg_return {sum(rets)/len(rets):.1f}')

C:\Users\pavan\AppData\Local\Temp\ipykernel_46652\4014515027.py:22: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  logp=Categorical(logits=pi(t.as_tensor(obs,dtype=t.float32))).log_prob(t.as_tensor(act))


epoch  0  avg_return 20.0


epoch 10  avg_return 232.0


epoch 20  avg_return 500.0


epoch 30  avg_return 440.2


epoch 40  avg_return 500.0


epoch 50  avg_return 232.4


epoch 59  avg_return 307.6
